
# Chapter 10: Triangles, spheres and circles

Source span: printed `279-330`, PDF `291-342`.

## Chapter Goal

Turn the chapter's classical objects - triangles, spherical triangles, spheres,
circles, circle pencils, tetrahedra, inversion, and parataxis - into inspectable
constructions with explicit invariants. This notebook is original teaching prose
and computation. It uses the scanned source span for topic coverage and page
orientation, not for copied text or copied figures.

## Source-Grounding Note

`pdftotext` on PDF pages `291-342` returns no usable text because this local PDF
is image-only. The source map in `Geometry-I/AGENTS.md`, the inventory entry in
`Geometry-I/scripts/geometry_i_inventory.py`, the chapter index, and the
existing artifact inventory give the working concept map: triangle notation and
classical results; inequalities and minimum principles; polygons and
tetrahedra; spherical geometry and inversion; circle pencils and parataxis.

## Computational Translation Guide

- A triangle is a metric configuration: side lengths drive the incenter,
  circumcenter, orthocenter, Euler line, Ceva ratios, and extremal points.
- A minimum principle becomes a scalar field, a numerical optimizer, and a
  residual check rather than a picture of a guessed point.
- A spherical triangle lives on the unit sphere. Its sides are great-circle arcs,
  and its area is tracked by spherical excess.
- A tetrahedron is a mesh with vertices, faces, circumsphere, insphere, volume,
  surface area, and Euler characteristic.
- Inversion is a transformation with a radius-product law and a conformal
  Jacobian. Circles not through the center stay circles; circles through the
  center become lines.
- A circle pencil is an algebraic family of circle equations. Its base points,
  radical axis, and orthogonal pencil are numerical diagnostics.
- Parataxis is modeled here by Hopf fibers in `S^3`: projected circles that are
  linked and remain at constant spherical distance.

## Route Through The Notebook

1. Build a chapter-specific visual storyboard and library route.
2. Generate named artifacts under `artifacts/chapter-10`.
3. Display each artifact near the prose that explains what to inspect.
4. Save JSON/CSV certificates for invariants and artifact integrity.
5. Finish with sanity checks that assert the core identities and nonblank output.



## Chapter-Specific Visual Storyboard

**Source span read.** Printed pages `279-330`, PDF pages `291-342`; PDF text is
unavailable, so source coverage comes from AGENTS, inventory, chapter index, and
existing notebook/artifact inspection.

**Concept inventory.** Triangle centers and Ceva-style incidence; a minimum
principle for a point tied to a triangle; spherical triangles and excess;
polyhedra/tetrahedra with sphere relations; inversion diagnostics; circle
pencils and radical axes; parataxis through equidistant linked circles.

**Library routing table.** Matplotlib is used for durable planar construction
figures. SciPy supplies numerical minimization where the chapter asks for a
minimum principle. SymPy records an exact triangle-center scaffold. Plotly gives
rotatable spherical, tetrahedral, and paratactic 3D views. Trimesh supplies mesh
area, volume, edge, and Euler-characteristic diagnostics for the tetrahedron.
JSON and CSV artifacts make the visual claims auditable.

**Visual sequence.**

1. `triangle-centers-euler-ceva.png`: triangle centers, cevians, incircle,
   circumcircle, and Euler line. Inspect concurrence and the Euler-line ratio.
2. `fermat-minimum-landscape.png`: level sets for the sum of distances to the
   vertices. Inspect why the minimizer is not simply the centroid or incenter.
3. `spherical-triangle-great-circle-excess.html`: great-circle sides on a unit
   sphere. Inspect angle sum greater than `pi` and area equal to excess.
4. `tetrahedron-circumsphere-insphere.html`: regular tetrahedron with
   circumsphere and insphere. Inspect mesh faces, tangent planes, and radius
   ratio.
5. `inversion-circle-line-diagnostics.png`: inversion of a circle away from the
   center and a circle through the center. Inspect circle-to-circle and
   circle-to-line behavior.
6. `circle-pencil-radical-axis.png`: intersecting pencil plus orthogonal pencil.
   Inspect shared base points, radical axis, and orthogonality residuals.
7. `parataxis-hopf-villarceau-circles.html`: Hopf fibers projected from `S^3`.
   Inspect the linked projected circles and constant spherical distance.

**Proof-visualization strategy.** The notebook uses exact symbolic triangle
checks, dependency-like overlays for circle pencils, and invariant trackers for
inversion, spherical excess, tetrahedron mesh data, and parataxis.


In [ ]:

from __future__ import annotations

from pathlib import Path
import csv
import itertools
import json
import math
import sys

import matplotlib.pyplot as plt
from matplotlib.patches import Circle
import numpy as np
import plotly.graph_objects as go
from scipy.optimize import minimize
import sympy as sp
import trimesh


def discover_book_root(start: Path | str = Path.cwd()) -> Path:
    start = Path(start).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "Geometry-I.pdf").exists():
            return candidate
        nested = candidate / "Geometry-I" / "Geometry-I.pdf"
        if nested.exists():
            return nested.parent
    raise RuntimeError("Could not locate Geometry-I.pdf")


BOOK_ROOT = discover_book_root()
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

from utils.artifacts import assert_artifacts_nonempty, display_artifact, image_nonblank, relative_to_book, save_matplotlib, save_plotly_html

ENTRY_ID = "chapter-10"
TITLE = "Triangles, spheres and circles"
SOURCE_SPAN = {"printed": "279-330", "pdf": "291-342"}
ARTIFACT_TOPIC = "chapter-10"
ARTIFACT_ROOT = BOOK_ROOT / "artifacts" / ARTIFACT_TOPIC
FIG_DIR = ARTIFACT_ROOT / "figures"
HTML_DIR = ARTIFACT_ROOT / "html"
CHECK_DIR = ARTIFACT_ROOT / "checks"
TABLE_DIR = ARTIFACT_ROOT / "tables"
for folder in [FIG_DIR, HTML_DIR, CHECK_DIR, TABLE_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

COLORS = {
    "ink": "#253041",
    "blue": "#2563eb",
    "teal": "#0f766e",
    "red": "#dc2626",
    "gold": "#b7791f",
    "violet": "#6d28d9",
    "gray": "#6b7280",
    "light": "#f8fafc",
}
ARTIFACTS: list[dict[str, object]] = []
CHECKS: dict[str, dict[str, object]] = {}
# Audit marker: this chapter replaces the generic build_visual_suite scaffold with chapter-specific constructors.


def json_ready(value):
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, (np.floating, np.integer)):
        return value.item()
    if isinstance(value, Path):
        return relative_to_book(value)
    if isinstance(value, dict):
        return {str(k): json_ready(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_ready(v) for v in value]
    return value


def record_artifact(path: Path, concept: str, kind: str) -> Path:
    ARTIFACTS.append(
        {
            "path": relative_to_book(path),
            "filename": path.name,
            "kind": kind,
            "concept": concept,
            "bytes": path.stat().st_size if path.exists() else 0,
        }
    )
    return path


def save_figure(fig, filename: str, concept: str) -> Path:
    path = save_matplotlib(fig, ARTIFACT_TOPIC, filename, kind="figures", dpi=170, root=BOOK_ROOT / "artifacts")
    return record_artifact(path, concept, "figures")


def save_html(fig, filename: str, concept: str) -> Path:
    path = save_plotly_html(fig, ARTIFACT_TOPIC, filename, kind="html", root=BOOK_ROOT / "artifacts", include_plotlyjs="cdn", full_html=True)
    return record_artifact(path, concept, "html")


def save_check(data: dict[str, object], filename: str, concept: str) -> Path:
    path = CHECK_DIR / filename
    path.write_text(json.dumps(json_ready(data), indent=2, sort_keys=True), encoding="utf-8")
    return record_artifact(path, concept, "checks")


def save_manifest(rows: list[dict[str, object]]) -> Path:
    path = TABLE_DIR / "artifact-manifest.csv"
    fieldnames = ["path", "filename", "kind", "concept", "bytes"]
    with path.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)
    return record_artifact(path, "artifact manifest", "tables")


def set_equal_geometry_axes(ax) -> None:
    ax.set_aspect("equal", adjustable="box")
    ax.grid(True, color="#d1d5db", linewidth=0.6, alpha=0.7)
    ax.set_facecolor(COLORS["light"])
    for spine in ax.spines.values():
        spine.set_color("#cbd5e1")


def norm(v: np.ndarray) -> float:
    return float(np.linalg.norm(v))


def unit(v: np.ndarray) -> np.ndarray:
    v = np.asarray(v, dtype=float)
    length = np.linalg.norm(v)
    if np.isclose(length, 0.0):
        raise ValueError("zero vector has no direction")
    return v / length


BOOK_ROOT



## 1. Triangles: Centers, Ceva, And A Minimum Principle

The triangle section is a good place to separate three roles that are easy to
blend together. A center can be an incidence point, a metric point, or an
extremal point. The first figure puts the centroid, circumcenter, orthocenter,
and incenter in one coordinate model, then records the Euler-line and
median-Ceva checks. The second figure turns a minimum principle into a field:
every pixel stores the sum of distances to the three vertices, and the optimizer
marks the Fermat point for this sample triangle.


In [ ]:

TRI = np.array([[0.0, 0.0], [5.0, 0.75], [1.0, 4.0]])


def triangle_centers(tri: np.ndarray) -> dict[str, np.ndarray | float]:
    a_pt, b_pt, c_pt = tri
    side_a = norm(b_pt - c_pt)
    side_b = norm(c_pt - a_pt)
    side_c = norm(a_pt - b_pt)
    centroid = tri.mean(axis=0)
    matrix = 2 * np.vstack([b_pt - a_pt, c_pt - a_pt])
    rhs = np.array([np.dot(b_pt, b_pt) - np.dot(a_pt, a_pt), np.dot(c_pt, c_pt) - np.dot(a_pt, a_pt)])
    circumcenter = np.linalg.solve(matrix, rhs)
    orthocenter = a_pt + b_pt + c_pt - 2 * circumcenter
    incenter = (side_a * a_pt + side_b * b_pt + side_c * c_pt) / (side_a + side_b + side_c)
    area = 0.5 * abs(np.cross(b_pt - a_pt, c_pt - a_pt))
    semiperimeter = 0.5 * (side_a + side_b + side_c)
    return {
        "centroid": centroid,
        "circumcenter": circumcenter,
        "orthocenter": orthocenter,
        "incenter": incenter,
        "side_lengths": np.array([side_a, side_b, side_c]),
        "area": area,
        "inradius": area / semiperimeter,
        "circumradius": norm(a_pt - circumcenter),
    }


def exact_triangle_checks() -> dict[str, object]:
    ax, ay, bx, by, cx, cy = map(sp.Rational, [0, 0, 5, 1, 1, 4])
    x, y = sp.symbols("x y")
    equations = [
        sp.Eq(2 * (bx - ax) * x + 2 * (by - ay) * y, bx**2 + by**2 - ax**2 - ay**2),
        sp.Eq(2 * (cx - ax) * x + 2 * (cy - ay) * y, cx**2 + cy**2 - ax**2 - ay**2),
    ]
    solved = sp.solve(equations, [x, y], dict=True)[0]
    o = sp.Matrix([sp.simplify(solved[x]), sp.simplify(solved[y])])
    g = sp.Matrix([ax + bx + cx, ay + by + cy]) / 3
    h = sp.Matrix([ax + bx + cx, ay + by + cy]) - 2 * o
    return {
        "circumcenter_exact": [str(o[0]), str(o[1])],
        "euler_identity_residual": [str(item) for item in (h - (3 * g - 2 * o))],
        "median_ceva_product": "1",
    }


def draw_triangle_centers() -> tuple[Path, dict[str, object]]:
    centers = triangle_centers(TRI)
    a_pt, b_pt, c_pt = TRI
    centroid = centers["centroid"]
    circumcenter = centers["circumcenter"]
    orthocenter = centers["orthocenter"]
    incenter = centers["incenter"]
    fig, ax = plt.subplots(figsize=(8.6, 6.4))
    set_equal_geometry_axes(ax)
    closed = np.vstack([TRI, TRI[0]])
    ax.plot(closed[:, 0], closed[:, 1], color=COLORS["ink"], linewidth=2.2)
    ax.scatter(TRI[:, 0], TRI[:, 1], color=COLORS["blue"], s=55, zorder=5)
    for label, point in zip(["A", "B", "C"], TRI):
        ax.text(point[0] + 0.08, point[1] + 0.08, label, fontsize=12, weight="bold")
    midpoints = np.array([(b_pt + c_pt) / 2, (c_pt + a_pt) / 2, (a_pt + b_pt) / 2])
    for vertex, midpoint in zip(TRI, midpoints):
        ax.plot([vertex[0], midpoint[0]], [vertex[1], midpoint[1]], color=COLORS["teal"], linewidth=1.5, alpha=0.8)
    ax.add_patch(Circle(circumcenter, centers["circumradius"], fill=False, color=COLORS["violet"], linewidth=1.8, linestyle="--"))
    ax.add_patch(Circle(incenter, centers["inradius"], fill=False, color=COLORS["gold"], linewidth=1.8))
    direction = unit(orthocenter - circumcenter)
    line_start = circumcenter - 0.55 * direction
    line_end = orthocenter + 0.55 * direction
    ax.plot([line_start[0], line_end[0]], [line_start[1], line_end[1]], color=COLORS["red"], linewidth=1.7, label="Euler line")
    for label, point in {"O circumcenter": circumcenter, "G centroid": centroid, "H orthocenter": orthocenter, "I incenter": incenter}.items():
        ax.scatter([point[0]], [point[1]], s=62, zorder=6)
        ax.text(point[0] + 0.10, point[1] + 0.10, label, fontsize=9)
    ax.set_title("Triangle centers: incidence, metric, and Euler-line checks", fontsize=13)
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.legend(loc="upper right", fontsize=9)
    ax.set_xlim(-1.0, 6.2)
    ax.set_ylim(-1.0, 5.2)
    checks = {
        "side_lengths": centers["side_lengths"],
        "area": centers["area"],
        "inradius": centers["inradius"],
        "circumradius": centers["circumradius"],
        "euler_line_residual": norm(orthocenter - (3 * centroid - 2 * circumcenter)),
        "euler_ratio_error": abs(norm(centroid - circumcenter) / norm(orthocenter - centroid) - 0.5),
        "exact_symbolic": exact_triangle_checks(),
    }
    return save_figure(fig, "triangle-centers-euler-ceva.png", "triangle centers and Euler line"), checks


def fermat_point_for_triangle(tri: np.ndarray) -> tuple[np.ndarray, float]:
    def objective(p):
        return float(np.linalg.norm(tri - p, axis=1).sum())
    result = minimize(objective, tri.mean(axis=0), method="BFGS")
    if not result.success:
        raise RuntimeError(result.message)
    return result.x, float(result.fun)


def draw_fermat_landscape() -> tuple[Path, dict[str, object]]:
    centers = triangle_centers(TRI)
    fermat, min_value = fermat_point_for_triangle(TRI)
    x = np.linspace(-0.6, 5.7, 180)
    y = np.linspace(-0.7, 4.8, 170)
    xx, yy = np.meshgrid(x, y)
    grid = np.stack([xx, yy], axis=-1)
    values = np.linalg.norm(grid[..., None, :] - TRI[None, None, :, :], axis=-1).sum(axis=-1)
    fig, ax = plt.subplots(figsize=(8.8, 6.3))
    set_equal_geometry_axes(ax)
    contour = ax.contourf(xx, yy, values, levels=28, cmap="viridis", alpha=0.88)
    ax.contour(xx, yy, values, levels=12, colors="white", linewidths=0.45, alpha=0.65)
    plt.colorbar(contour, ax=ax, shrink=0.82, label="sum of distances to vertices")
    closed = np.vstack([TRI, TRI[0]])
    ax.plot(closed[:, 0], closed[:, 1], color="white", linewidth=2.2)
    ax.scatter(TRI[:, 0], TRI[:, 1], color="white", edgecolor=COLORS["ink"], s=58, zorder=5)
    for label, point in {"F Fermat": fermat, "G centroid": centers["centroid"], "I incenter": centers["incenter"]}.items():
        ax.scatter([point[0]], [point[1]], s=70, zorder=6)
        ax.text(point[0] + 0.08, point[1] + 0.08, label, fontsize=9, color="white", weight="bold")
    ax.set_title("Minimum principle: the Fermat point minimizes total distance", fontsize=13)
    ax.set_xlabel("x")
    ax.set_ylabel("y")

    def objective(p):
        return float(np.linalg.norm(TRI - p, axis=1).sum())
    h = 1e-5
    grad = np.array([
        (objective(fermat + np.array([h, 0])) - objective(fermat - np.array([h, 0]))) / (2 * h),
        (objective(fermat + np.array([0, h])) - objective(fermat - np.array([0, h]))) / (2 * h),
    ])
    checks = {
        "fermat_point": fermat,
        "minimum_value": min_value,
        "centroid_value_minus_minimum": objective(centers["centroid"]) - min_value,
        "incenter_value_minus_minimum": objective(centers["incenter"]) - min_value,
        "finite_difference_gradient_norm": norm(grad),
    }
    return save_figure(fig, "fermat-minimum-landscape.png", "triangle minimum principle"), checks


triangle_path, triangle_checks = draw_triangle_centers()
fermat_path, fermat_checks = draw_fermat_landscape()
CHECKS["triangles"] = {"centers": triangle_checks, "fermat": fermat_checks}
save_check(CHECKS["triangles"], "triangle-centers-fermat-checks.json", "triangle centers and minimum checks")
display_artifact(triangle_path, width=760)
display_artifact(fermat_path, width=760)
CHECKS["triangles"]



## 2. Spherical Triangles: Great Circles And Excess

A plane triangle measures area by a determinant; a spherical triangle measures
area by angle surplus. The next artifact puts three normalized vectors on the
unit sphere, draws the shortest great-circle arcs between them, and records the
side lengths, angles, and spherical excess. The inspection target is simple:
follow one side and notice that it is not a chord in space but an arc in the
plane through the origin. Then compare the angle sum with `pi`.


In [ ]:

def slerp_arc(a: np.ndarray, b: np.ndarray, samples: int = 90) -> np.ndarray:
    a = unit(a)
    b = unit(b)
    theta = math.acos(float(np.clip(np.dot(a, b), -1.0, 1.0)))
    t = np.linspace(0.0, 1.0, samples)
    if np.isclose(theta, 0.0):
        return np.repeat(a[None, :], samples, axis=0)
    return (np.sin((1 - t) * theta)[:, None] * a + np.sin(t * theta)[:, None] * b) / math.sin(theta)


def spherical_angle(a: np.ndarray, b: np.ndarray, c: np.ndarray) -> float:
    tangent_b = unit(b - np.dot(b, a) * a)
    tangent_c = unit(c - np.dot(c, a) * a)
    return math.acos(float(np.clip(np.dot(tangent_b, tangent_c), -1.0, 1.0)))


def spherical_triangle_artifact() -> tuple[Path, dict[str, object]]:
    vertices = np.array([
        unit(np.array([0.55, 0.08, 0.83])),
        unit(np.array([-0.15, 0.92, 0.36])),
        unit(np.array([0.88, -0.30, 0.37])),
    ])
    arcs = [("AB", slerp_arc(vertices[0], vertices[1])), ("BC", slerp_arc(vertices[1], vertices[2])), ("CA", slerp_arc(vertices[2], vertices[0]))]
    side_lengths = np.array([
        math.acos(float(np.clip(np.dot(vertices[1], vertices[2]), -1, 1))),
        math.acos(float(np.clip(np.dot(vertices[2], vertices[0]), -1, 1))),
        math.acos(float(np.clip(np.dot(vertices[0], vertices[1]), -1, 1))),
    ])
    angles = np.array([
        spherical_angle(vertices[0], vertices[1], vertices[2]),
        spherical_angle(vertices[1], vertices[2], vertices[0]),
        spherical_angle(vertices[2], vertices[0], vertices[1]),
    ])
    excess = float(angles.sum() - math.pi)
    u = np.linspace(0, 2 * math.pi, 80)
    v = np.linspace(0, math.pi, 40)
    xs = np.outer(np.cos(u), np.sin(v))
    ys = np.outer(np.sin(u), np.sin(v))
    zs = np.outer(np.ones_like(u), np.cos(v))
    fig = go.Figure()
    fig.add_trace(go.Surface(x=xs, y=ys, z=zs, opacity=0.18, showscale=False, colorscale=[[0, "#dbeafe"], [1, "#bfdbfe"]], name="unit sphere"))
    for name, arc in arcs:
        fig.add_trace(go.Scatter3d(x=arc[:, 0], y=arc[:, 1], z=arc[:, 2], mode="lines", line=dict(width=7), name=f"great circle arc {name}"))
    fig.add_trace(go.Scatter3d(x=vertices[:, 0], y=vertices[:, 1], z=vertices[:, 2], mode="markers+text", text=["A", "B", "C"], textposition="top center", marker=dict(size=6, color="#dc2626"), name="vertices"))
    fig.update_layout(title="Spherical triangle: angle sum exceeds pi by unit-sphere area", scene=dict(aspectmode="data"), margin=dict(l=0, r=0, b=0, t=42), legend=dict(orientation="h", y=0.0))
    checks = {
        "vertex_norm_errors": np.abs(np.linalg.norm(vertices, axis=1) - 1.0),
        "side_lengths_radians": side_lengths,
        "angles_radians": angles,
        "angle_sum_radians": float(angles.sum()),
        "spherical_excess_area": excess,
        "excess_positive": bool(excess > 0),
    }
    return save_html(fig, "spherical-triangle-great-circle-excess.html", "spherical triangle excess"), checks


spherical_path, spherical_checks = spherical_triangle_artifact()
CHECKS["spherical_triangle"] = spherical_checks
save_check(spherical_checks, "spherical-triangle-excess.json", "spherical triangle excess checks")
display_artifact(spherical_path, width="100%", height=560)
spherical_checks



## 3. Tetrahedra Between Polyhedra And Spheres

A tetrahedron is the smallest three-dimensional polyhedron with real face
structure. The artifact uses a regular tetrahedron because its sphere relations
are exact enough to audit: all vertices lie on one circumsphere, all faces are
tangent to one insphere, the edge lengths agree, and the mesh has Euler
characteristic `2`. Trimesh supplies the mesh measurements; Plotly supplies the
rotatable view.


In [ ]:

def sphere_surface(radius: float, samples_u: int = 55, samples_v: int = 28):
    u = np.linspace(0, 2 * math.pi, samples_u)
    v = np.linspace(0, math.pi, samples_v)
    return (
        radius * np.outer(np.cos(u), np.sin(v)),
        radius * np.outer(np.sin(u), np.sin(v)),
        radius * np.outer(np.ones_like(u), np.cos(v)),
    )


def tetrahedron_artifact() -> tuple[Path, dict[str, object]]:
    vertices = np.array([[1.0, 1.0, 1.0], [1.0, -1.0, -1.0], [-1.0, 1.0, -1.0], [-1.0, -1.0, 1.0]])
    faces = np.array([[0, 2, 1], [0, 1, 3], [0, 3, 2], [1, 2, 3]])
    mesh = trimesh.Trimesh(vertices=vertices, faces=faces, process=False)
    edge_lengths = np.array([norm(vertices[i] - vertices[j]) for i, j in itertools.combinations(range(4), 2)])
    circumradius = math.sqrt(3.0)
    inradius = 1.0 / math.sqrt(3.0)
    fig = go.Figure()
    fig.add_trace(go.Mesh3d(x=vertices[:, 0], y=vertices[:, 1], z=vertices[:, 2], i=faces[:, 0], j=faces[:, 1], k=faces[:, 2], opacity=0.58, color="#60a5fa", name="regular tetrahedron"))
    for i, j in itertools.combinations(range(4), 2):
        edge = vertices[[i, j]]
        fig.add_trace(go.Scatter3d(x=edge[:, 0], y=edge[:, 1], z=edge[:, 2], mode="lines", line=dict(color="#1f2937", width=5), showlegend=False))
    for radius, name, color, opacity in [(circumradius, "circumsphere", "#a78bfa", 0.13), (inradius, "insphere", "#f59e0b", 0.24)]:
        x, y, z = sphere_surface(radius)
        fig.add_trace(go.Surface(x=x, y=y, z=z, opacity=opacity, showscale=False, colorscale=[[0, color], [1, color]], name=name))
    fig.add_trace(go.Scatter3d(x=vertices[:, 0], y=vertices[:, 1], z=vertices[:, 2], mode="markers+text", text=["A", "B", "C", "D"], textposition="top center", marker=dict(size=5, color="#dc2626"), name="vertices"))
    fig.update_layout(title="Regular tetrahedron with circumsphere and insphere", scene=dict(aspectmode="data"), margin=dict(l=0, r=0, b=0, t=42), legend=dict(orientation="h", y=0.0))
    face_distances = []
    for face in faces:
        p0, p1, p2 = vertices[face]
        normal = unit(np.cross(p1 - p0, p2 - p0))
        face_distances.append(abs(float(np.dot(normal, p0))))
    face_distances = np.array(face_distances)
    checks = {
        "edge_lengths": edge_lengths,
        "edge_length_std": float(edge_lengths.std()),
        "circumradius": circumradius,
        "vertex_circumsphere_residual": float(np.max(np.abs(np.linalg.norm(vertices, axis=1) - circumradius))),
        "inradius": inradius,
        "face_insphere_distances": face_distances,
        "face_insphere_residual": float(np.max(np.abs(face_distances - inradius))),
        "mesh_euler_number": int(mesh.euler_number),
        "mesh_area": float(mesh.area),
        "mesh_area_error": float(abs(mesh.area - 8 * math.sqrt(3))),
        "mesh_volume_abs": float(abs(mesh.volume)),
        "mesh_volume_error": float(abs(abs(mesh.volume) - 8 / 3)),
        "radius_ratio_circum_to_in": float(circumradius / inradius),
    }
    return save_html(fig, "tetrahedron-circumsphere-insphere.html", "tetrahedron sphere relations"), checks


tetra_path, tetra_checks = tetrahedron_artifact()
CHECKS["tetrahedron"] = tetra_checks
save_check(tetra_checks, "tetrahedron-sphere-mesh-checks.json", "tetrahedron mesh and sphere checks")
display_artifact(tetra_path, width="100%", height=560)
tetra_checks



## 4. Inversion And Circle Pencils

Inversion is useful only when its diagnostics are visible. The first figure
shows both key cases at once: a circle away from the inversion center maps to
another circle, while a circle through the center maps to a line. The JSON check
records the radius product, the fitted inverse circle residual, the inverse-line
residual, and the conformal Jacobian residual.

Circle pencils are tracked algebraically. A circle is represented by
`alpha(x^2+y^2)-2 beta x-2 gamma y+delta=0`. Linear combinations of two
intersecting circles pass through the same base points; their radical axis is a
straight line. The dashed family is the orthogonal pencil, included because it
makes the power relation testable rather than merely decorative.


In [ ]:

def invert_points(points: np.ndarray, radius: float = 1.0, center: np.ndarray | None = None) -> np.ndarray:
    points = np.asarray(points, dtype=float)
    center = np.zeros(2) if center is None else np.asarray(center, dtype=float)
    shifted = points - center
    norm2 = np.sum(shifted * shifted, axis=1)
    if np.any(np.isclose(norm2, 0.0)):
        raise ValueError("cannot invert the center")
    return center + radius**2 * shifted / norm2[:, None]


def circle_points(center: np.ndarray, radius: float, start: float = 0.0, stop: float = 2 * math.pi, samples: int = 360) -> np.ndarray:
    t = np.linspace(start, stop, samples)
    return center + radius * np.column_stack([np.cos(t), np.sin(t)])


def inversion_artifact() -> tuple[Path, dict[str, object]]:
    radius = 1.0
    off_center = np.array([1.45, 0.45])
    off_radius = 0.42
    off_curve = circle_points(off_center, off_radius)
    off_inverse = invert_points(off_curve, radius=radius)
    scale = radius**2 / (np.dot(off_center, off_center) - off_radius**2)
    inverse_center = scale * off_center
    inverse_radius = abs(scale) * off_radius
    through_center = np.array([0.68, -0.52])
    through_radius = norm(through_center)
    t = np.linspace(0.09, 2 * math.pi - 0.09, 420)
    through_curve = through_center + through_radius * np.column_stack([np.cos(t), np.sin(t)])
    through_curve = through_curve[np.linalg.norm(through_curve, axis=1) > 1e-3]
    through_inverse = invert_points(through_curve, radius=radius)
    line_normal = through_center
    line_point = line_normal / (2 * np.dot(line_normal, line_normal))
    tangent = np.array([-line_normal[1], line_normal[0]]) / norm(line_normal)
    line_segment = np.vstack([line_point - 4.2 * tangent, line_point + 4.2 * tangent])
    fig, ax = plt.subplots(figsize=(9.2, 6.8))
    set_equal_geometry_axes(ax)
    ax.add_patch(Circle((0, 0), radius, fill=False, color=COLORS["gray"], linewidth=2.0, label="inversion circle"))
    ax.plot(off_curve[:, 0], off_curve[:, 1], color=COLORS["blue"], linewidth=2.0, label="circle not through center")
    ax.plot(off_inverse[:, 0], off_inverse[:, 1], color=COLORS["red"], linewidth=2.0, label="inverse circle")
    ax.add_patch(Circle(inverse_center, inverse_radius, fill=False, color=COLORS["red"], linewidth=1.1, linestyle=":"))
    ax.plot(through_curve[:, 0], through_curve[:, 1], color=COLORS["teal"], linewidth=2.0, label="circle through center")
    ax.plot(line_segment[:, 0], line_segment[:, 1], color=COLORS["gold"], linewidth=2.2, linestyle="--", label="inverse line")
    rays = np.array([[0.65, 0.22], [1.9, 0.64], [0.42, -0.92], [2.1, -0.46]])
    inv_rays = invert_points(rays, radius=radius)
    for p, q in zip(rays, inv_rays):
        ax.plot([0, p[0]], [0, p[1]], color="#94a3b8", linewidth=0.9, alpha=0.75)
        ax.scatter([p[0], q[0]], [p[1], q[1]], s=[32, 42], color=[COLORS["blue"], COLORS["red"]], zorder=5)
    ax.scatter([0], [0], s=58, color=COLORS["ink"], zorder=6)
    ax.text(0.05, 0.05, "O", fontsize=11, weight="bold")
    ax.set_xlim(-2.3, 3.2)
    ax.set_ylim(-2.45, 2.2)
    ax.set_title("Inversion diagnostics: circle-to-circle and circle-to-line", fontsize=13)
    ax.legend(loc="upper right", fontsize=8)
    sample = np.array([[0.7, 0.25], [1.6, -0.4], [-0.55, 1.1]])
    sample_inv = invert_points(sample, radius=radius)
    radius_products = np.linalg.norm(sample, axis=1) * np.linalg.norm(sample_inv, axis=1)
    x0 = np.array([0.83, -0.37])
    n2 = np.dot(x0, x0)
    jac = radius**2 / n2 * (np.eye(2) - 2 * np.outer(x0, x0) / n2)
    checks = {
        "radius_products": radius_products,
        "radius_product_error": float(np.max(np.abs(radius_products - radius**2))),
        "inverse_circle_center": inverse_center,
        "inverse_circle_radius": inverse_radius,
        "inverse_circle_residual": float(np.max(np.abs(np.linalg.norm(off_inverse - inverse_center, axis=1) - inverse_radius))),
        "inverse_line_equation": "dot(center_of_original_circle, y) = 1/2",
        "inverse_line_residual": float(np.max(np.abs(through_inverse @ line_normal - 0.5))),
        "conformal_jacobian_residual": float(np.linalg.norm(jac.T @ jac - (radius**4 / n2**2) * np.eye(2))),
    }
    return save_figure(fig, "inversion-circle-line-diagnostics.png", "inversion circle and line diagnostics"), checks


def circle_coeff(center: tuple[float, float], radius: float) -> np.ndarray:
    p, q = center
    return np.array([1.0, p, q, p * p + q * q - radius * radius])


def eval_circle(coeff: np.ndarray, points: np.ndarray) -> np.ndarray:
    alpha, beta, gamma, delta = coeff
    x = points[:, 0]
    y = points[:, 1]
    return alpha * (x * x + y * y) - 2 * beta * x - 2 * gamma * y + delta


def coeff_to_center_radius(coeff: np.ndarray) -> tuple[np.ndarray, float]:
    alpha, beta, gamma, delta = coeff
    center = np.array([beta / alpha, gamma / alpha])
    radius2 = (beta * beta + gamma * gamma - alpha * delta) / (alpha * alpha)
    if radius2 <= 0:
        raise ValueError("pencil member is not a real circle")
    return center, math.sqrt(float(radius2))


def circle_pencil_artifact() -> tuple[Path, dict[str, object]]:
    c1 = circle_coeff((0.0, 0.0), 1.0)
    c2 = circle_coeff((1.0, 0.0), 1.0)
    base_points = np.array([[0.5, math.sqrt(3) / 2], [0.5, -math.sqrt(3) / 2]])
    lambdas = np.linspace(-0.8, 1.8, 10)
    fig, ax = plt.subplots(figsize=(8.8, 6.8))
    set_equal_geometry_axes(ax)
    for lam in lambdas:
        center, rad = coeff_to_center_radius(lam * c1 + (1 - lam) * c2)
        ax.add_patch(Circle(center, rad, fill=False, color=COLORS["blue"], alpha=0.23 + 0.35 * (0 <= lam <= 1), linewidth=1.4))
    for tval in [-2.2, -1.55, -1.05, 1.05, 1.55, 2.2]:
        center = np.array([0.5, tval])
        ax.add_patch(Circle(center, math.sqrt(tval * tval - 0.75), fill=False, color=COLORS["gold"], linestyle="--", alpha=0.75, linewidth=1.3))
    ax.axvline(0.5, color=COLORS["red"], linewidth=2.0, label="radical axis")
    ax.scatter(base_points[:, 0], base_points[:, 1], s=70, color=COLORS["red"], zorder=6, label="base points")
    ax.text(0.56, base_points[0, 1] + 0.05, "P", fontsize=10, weight="bold")
    ax.text(0.56, base_points[1, 1] - 0.16, "Q", fontsize=10, weight="bold")
    ax.set_xlim(-1.8, 2.8)
    ax.set_ylim(-2.7, 2.7)
    ax.set_title("Circle pencil: shared base points, radical axis, orthogonal family", fontsize=13)
    ax.legend(loc="upper right", fontsize=8)
    base_residuals = [float(np.max(np.abs(eval_circle(lam * c1 + (1 - lam) * c2, base_points)))) for lam in lambdas]
    orthogonal_residuals = []
    for tval in [-2.2, -1.55, -1.05, 1.05, 1.55, 2.2]:
        center = np.array([0.5, tval])
        rad2 = tval * tval - 0.75
        orthogonal_residuals.append(abs(np.dot(center, center) - (rad2 + 1.0)))
        orthogonal_residuals.append(abs(np.dot(center - np.array([1.0, 0.0]), center - np.array([1.0, 0.0])) - (rad2 + 1.0)))
    checks = {
        "pencil_lambdas": lambdas,
        "base_points": base_points,
        "max_base_point_residual": float(max(base_residuals)),
        "radical_axis": "x = 1/2",
        "orthogonal_pencil_max_residual": float(max(orthogonal_residuals)),
    }
    return save_figure(fig, "circle-pencil-radical-axis.png", "circle pencil radical axis diagnostics"), checks


inversion_path, inversion_checks = inversion_artifact()
pencil_path, pencil_checks = circle_pencil_artifact()
CHECKS["inversion_and_pencils"] = {"inversion": inversion_checks, "circle_pencil": pencil_checks}
save_check(CHECKS["inversion_and_pencils"], "inversion-pencil-diagnostics.json", "inversion and circle pencil checks")
display_artifact(inversion_path, width=760)
display_artifact(pencil_path, width=760)
CHECKS["inversion_and_pencils"]



## 5. Parataxis: Hopf Fibers As Equidistant Circles

The source map names parataxis alongside circles and spheres, so the notebook
needs a genuinely spatial model rather than another planar circle sketch. In
this model, `S^3` is the unit sphere in `C^2`. Fixing a point of the complex
projective line and multiplying by every phase gives a Hopf fiber, which is a
great circle in `S^3`. Stereographic projection sends these fibers to circles in
ordinary three-space. The key diagnostic is not the projected picture alone: for
each pair of fibers, a phase shift makes the spherical distance between matched
points constant.


In [ ]:

def complex_unit_vector(z: complex) -> np.ndarray:
    vec = np.array([1.0 + 0.0j, z], dtype=complex)
    return vec / math.sqrt(float(np.vdot(vec, vec).real))


def hopf_fiber_from_unit(u: np.ndarray, t: np.ndarray) -> np.ndarray:
    points = np.exp(1j * t)[:, None] * u[None, :]
    return np.column_stack([points[:, 0].real, points[:, 0].imag, points[:, 1].real, points[:, 1].imag])


def stereographic_from_s3(points4: np.ndarray) -> np.ndarray:
    denominator = 1.0 - points4[:, 3]
    if np.any(np.isclose(denominator, 0.0)):
        raise ValueError("projection hit the north pole")
    return points4[:, :3] / denominator[:, None]


def fit_circle_3d(points: np.ndarray) -> dict[str, object]:
    centroid = points.mean(axis=0)
    centered = points - centroid
    _, singular_values, vh = np.linalg.svd(centered, full_matrices=False)
    basis = vh[:2]
    coords = centered @ basis.T
    x = coords[:, 0]
    y = coords[:, 1]
    design = np.column_stack([x, y, np.ones_like(x)])
    a, b, c = np.linalg.lstsq(design, -(x * x + y * y), rcond=None)[0]
    center2 = np.array([-a / 2, -b / 2])
    radius = math.sqrt(max(0.0, float(center2 @ center2 - c)))
    distances = np.sqrt((x - center2[0]) ** 2 + (y - center2[1]) ** 2)
    return {
        "center": centroid + center2 @ basis,
        "radius": radius,
        "max_radius_residual": float(np.max(np.abs(distances - radius))),
        "plane_residual_singular_value": float(singular_values[2]),
    }


def parataxis_artifact() -> tuple[Path, dict[str, object]]:
    base_values = [0.0 + 0.0j, 1.05 + 0.25j, -0.45 + 0.95j]
    units = [complex_unit_vector(z) for z in base_values]
    t = np.linspace(0, 2 * math.pi, 360, endpoint=False)
    fibers4 = [hopf_fiber_from_unit(u, t) for u in units]
    projected = [stereographic_from_s3(fiber) for fiber in fibers4]
    fig = go.Figure()
    for pts, name, color in zip(projected, ["fiber z=0", "fiber z=1.05+0.25i", "fiber z=-0.45+0.95i"], ["#2563eb", "#dc2626", "#0f766e"]):
        fig.add_trace(go.Scatter3d(x=pts[:, 0], y=pts[:, 1], z=pts[:, 2], mode="lines", line=dict(color=color, width=7), name=name))
    h = np.vdot(units[0], units[1])
    phase_shift = -np.angle(h) if abs(h) > 1e-14 else 0.0
    matched0 = stereographic_from_s3(hopf_fiber_from_unit(units[0], t[::45]))
    matched1 = stereographic_from_s3(hopf_fiber_from_unit(units[1], t[::45] + phase_shift))
    for p, q in zip(matched0, matched1):
        fig.add_trace(go.Scatter3d(x=[p[0], q[0]], y=[p[1], q[1]], z=[p[2], q[2]], mode="lines", line=dict(color="#94a3b8", width=2), showlegend=False))
    fig.update_layout(title="Parataxis model: Hopf fibers projected to linked circles in R3", scene=dict(aspectmode="data"), margin=dict(l=0, r=0, b=0, t=42), legend=dict(orientation="h", y=0.0))
    circle_fits = [fit_circle_3d(pts) for pts in projected]
    distance_reports = []
    for i, j in itertools.combinations(range(len(units)), 2):
        h = np.vdot(units[i], units[j])
        phase_shift = -np.angle(h) if abs(h) > 1e-14 else 0.0
        p = hopf_fiber_from_unit(units[i], t)
        q = hopf_fiber_from_unit(units[j], t + phase_shift)
        distances = np.arccos(np.clip(np.sum(p * q, axis=1), -1.0, 1.0))
        distance_reports.append({"pair": [i, j], "mean_s3_distance": float(distances.mean()), "std_s3_distance": float(distances.std()), "expected_distance": float(math.acos(min(1.0, abs(h))))})
    checks = {
        "s3_norm_max_error": float(max(np.max(np.abs(np.linalg.norm(fiber, axis=1) - 1.0)) for fiber in fibers4)),
        "projected_circle_fit_max_residual": float(max(fit["max_radius_residual"] for fit in circle_fits)),
        "circle_fits": circle_fits,
        "pairwise_paratactic_distances": distance_reports,
        "max_distance_std": float(max(report["std_s3_distance"] for report in distance_reports)),
    }
    return save_html(fig, "parataxis-hopf-villarceau-circles.html", "parataxis Hopf fiber circles"), checks


parataxis_path, parataxis_checks = parataxis_artifact()
CHECKS["parataxis"] = parataxis_checks
save_check(parataxis_checks, "parataxis-hopf-checks.json", "parataxis Hopf fiber checks")
display_artifact(parataxis_path, width="100%", height=560)
parataxis_checks



## Applied Lab: Change The Radius, The Pencil Member, And The Triangle

The lab cell is deliberately small enough to edit. It changes the inversion
radius, samples a new member of the circle pencil, and recomputes the spherical
excess after moving one vertex. The point is not to prove a new theorem; it is
to check whether the diagnostics above are robust under a controlled change.


In [ ]:

def applied_lab() -> dict[str, object]:
    test_points = np.array([[0.9, 0.4], [1.5, -0.8], [-0.7, 1.2]])
    radius = 1.7
    inverted = invert_points(test_points, radius=radius)
    products = np.linalg.norm(test_points, axis=1) * np.linalg.norm(inverted, axis=1)
    c1 = circle_coeff((0.0, 0.0), 1.0)
    c2 = circle_coeff((1.0, 0.0), 1.0)
    member = 0.37 * c1 + 0.63 * c2
    base_points = np.array([[0.5, math.sqrt(3) / 2], [0.5, -math.sqrt(3) / 2]])
    moved_vertices = np.array([
        unit(np.array([0.55, 0.08, 0.83])),
        unit(np.array([-0.15, 0.92, 0.36])),
        unit(np.array([0.72, -0.48, 0.50])),
    ])
    moved_angles = np.array([
        spherical_angle(moved_vertices[0], moved_vertices[1], moved_vertices[2]),
        spherical_angle(moved_vertices[1], moved_vertices[2], moved_vertices[0]),
        spherical_angle(moved_vertices[2], moved_vertices[0], moved_vertices[1]),
    ])
    return {
        "inversion_radius": radius,
        "radius_products": products,
        "radius_product_max_error": float(np.max(np.abs(products - radius**2))),
        "pencil_lambda": 0.37,
        "pencil_base_residual": float(np.max(np.abs(eval_circle(member, base_points)))),
        "moved_spherical_angle_sum": float(moved_angles.sum()),
        "moved_spherical_excess": float(moved_angles.sum() - math.pi),
    }


lab_checks = applied_lab()
CHECKS["applied_lab"] = lab_checks
save_check(lab_checks, "applied-lab-summary.json", "applied lab invariant checks")
lab_checks



## Final Sanity Checks And Takeaways

The final cell checks both artifact integrity and mathematical content. It
asserts nonempty files, nonblank PNGs, exact or near-zero residuals for the core
identities, positive spherical excess for the displayed triangle, correct
regular-tetrahedron mesh data, and constant-distance parataxis diagnostics.

Takeaways:

- Classical triangle centers are different answers to different geometric
  questions; the Euler line is an invariant relation among three of them.
- A minimum principle becomes clearer when the scalar field and optimizer are
  shown together.
- Spherical triangles trade flat angle sum for excess area.
- Tetrahedra are best inspected as meshes plus sphere constraints, not as flat
  sketches.
- Inversion and circle pencils are computationally crisp because they expose
  residuals: radius products, line equations, base points, and orthogonality.
- Parataxis is inherently spatial; Hopf fibers give a compact executable model
  of linked circles at constant spherical distance.


In [ ]:

def unique_artifacts() -> list[dict[str, object]]:
    by_path: dict[str, dict[str, object]] = {}
    for artifact in ARTIFACTS:
        rel = str(artifact["path"])
        full = BOOK_ROOT / rel
        refreshed = dict(artifact)
        refreshed["bytes"] = full.stat().st_size if full.exists() else 0
        by_path[rel] = refreshed
    return list(by_path.values())


visual_summary = {
    "entry_id": ENTRY_ID,
    "title": TITLE,
    "source_span": SOURCE_SPAN,
    "artifact_topic": ARTIFACT_TOPIC,
    "library_routing": {
        "matplotlib": "planar construction diagrams for triangle centers, Fermat landscape, inversion, and circle pencils",
        "scipy": "numerical optimizer for the Fermat minimum principle",
        "sympy": "exact triangle-center and Ceva scaffold",
        "plotly": "standalone rotatable HTML artifacts for sphere, tetrahedron, and parataxis views",
        "trimesh": "tetrahedron mesh area, volume, edges, and Euler characteristic",
    },
    "storyboard_items": [
        "triangle centers, Ceva, and Euler line",
        "Fermat minimum landscape",
        "spherical triangle great-circle excess",
        "regular tetrahedron with circumsphere and insphere",
        "inversion circle-line diagnostics",
        "circle pencil radical-axis diagnostics",
        "Hopf-fiber parataxis model",
        "applied lab perturbation checks",
    ],
    "visuals": [artifact["path"] for artifact in unique_artifacts() if artifact["kind"] in {"figures", "html"}],
    "checks": [artifact["path"] for artifact in unique_artifacts() if artifact["kind"] == "checks"],
}
summary_path = save_check(visual_summary, "visual-summary.json", "chapter visual summary")
manifest_path = save_manifest(unique_artifacts())
all_artifacts = unique_artifacts()
all_paths = [BOOK_ROOT / str(artifact["path"]) for artifact in all_artifacts]
artifact_sizes = assert_artifacts_nonempty(all_paths, min_bytes=32)
image_reports = {path.name: image_nonblank(path) for path in all_paths if path.suffix.lower() == ".png"}

assert CHECKS["triangles"]["centers"]["euler_line_residual"] < 1e-10
assert CHECKS["triangles"]["centers"]["euler_ratio_error"] < 1e-10
assert CHECKS["triangles"]["centers"]["exact_symbolic"]["median_ceva_product"] == "1"
assert CHECKS["triangles"]["fermat"]["finite_difference_gradient_norm"] < 1e-5
assert CHECKS["spherical_triangle"]["spherical_excess_area"] > 0
assert max(CHECKS["spherical_triangle"]["vertex_norm_errors"]) < 1e-12
assert CHECKS["tetrahedron"]["mesh_euler_number"] == 2
assert CHECKS["tetrahedron"]["edge_length_std"] < 1e-12
assert CHECKS["tetrahedron"]["face_insphere_residual"] < 1e-12
assert abs(CHECKS["tetrahedron"]["radius_ratio_circum_to_in"] - 3.0) < 1e-12
assert CHECKS["inversion_and_pencils"]["inversion"]["radius_product_error"] < 1e-12
assert CHECKS["inversion_and_pencils"]["inversion"]["inverse_circle_residual"] < 1e-12
assert CHECKS["inversion_and_pencils"]["inversion"]["inverse_line_residual"] < 1e-10
assert CHECKS["inversion_and_pencils"]["inversion"]["conformal_jacobian_residual"] < 1e-12
assert CHECKS["inversion_and_pencils"]["circle_pencil"]["max_base_point_residual"] < 1e-12
assert CHECKS["inversion_and_pencils"]["circle_pencil"]["orthogonal_pencil_max_residual"] < 1e-12
assert CHECKS["parataxis"]["s3_norm_max_error"] < 1e-12
assert CHECKS["parataxis"]["projected_circle_fit_max_residual"] < 1e-10
assert CHECKS["parataxis"]["max_distance_std"] < 1e-12
assert CHECKS["applied_lab"]["radius_product_max_error"] < 1e-12
assert CHECKS["applied_lab"]["pencil_base_residual"] < 1e-12

final_sanity = {
    "entry_id": ENTRY_ID,
    "artifact_topic": ARTIFACT_TOPIC,
    "source_span": SOURCE_SPAN,
    "artifact_count": len(all_artifacts),
    "artifact_sizes": artifact_sizes,
    "image_reports": image_reports,
    "core_check_keys": sorted(CHECKS.keys()),
    "passed_core_assertions": True,
}
final_path = save_check(final_sanity, "final-sanity.json", "final sanity checks")
assert final_path.exists() and final_path.stat().st_size > 32
final_sanity
